In [26]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [27]:
movies = pd.read_csv("C:\\Users\\Sonali\\Downloads\\Movie_Project\\tmdb_5000_movies.csv")
credits = pd.read_csv("C:\\Users\\Sonali\\Downloads\\Movie_Project\\tmdb_5000_credits.csv")

In [ ]:
movies = movies.merge(credits,on='title')  

In [29]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [30]:
movies.dropna(inplace=True)

In [31]:
def convert(text):
    L = []
    
    if isinstance(text, str):
        for i in ast.literal_eval(text):
            L.append(i['name'])
            
    return L

In [32]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [33]:
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
    return L

In [34]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [35]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [36]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [37]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [38]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [39]:
new_df = movies[['movie_id','title','tags']].copy()

In [40]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [41]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x) if isinstance(x, list) else "")

In [42]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

In [43]:
new_df['tags'].head()

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: tags, dtype: object

In [44]:
similarity = cosine_similarity(vectors)

In [45]:
pickle.dump(new_df.to_dict(), open('movies_dict.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))